In [4]:
!pip3 install scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 3.5 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 662.2 kB/s eta 0:00:00m eta 0:00:010:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 KB 1.3 MB/s eta 0:00:00m eta 0:00:010:00:01


In [1]:
import numpy as np 
import cv2
import pandas as pd 
import os
import pickle


In [2]:
face_detection_model= './models/res10_300x300_ssd_iter_140000.caffemodel'
face_detection_proto= './models/deploy.prototxt.txt'
face_descriptor= './models/nn4.small2.v1.t7'

detector_model= cv2.dnn.readNetFromCaffe(face_detection_proto,face_detection_model)
descriptor_model= cv2.dnn.readNetFromTorch(face_descriptor)
face_recognition_model= pickle.load(open('./models/machinelearning_face_person_identiti.pkl',mode='rb'))
emotion_recognition_model= pickle.load(open('./models/machinelearning_emotions.pkl',mode='rb'))

In [7]:
img =cv2.imread('./images/Robert_Downey_Jr/005_8af3cada.jpg')
# cv2.imshow("img",img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [8]:
def helper(img_path):
    img =cv2.imread(img_path)
    imgs= img.copy()
    h,w= imgs.shape[:2]
    if h>1500 and w>1500:
        imgs=cv2.resize(img,None,fx=0.2,fy=0.2,interpolation=cv2.INTER_AREA)
        h,w= imgs.shape[:2]

    img_blob= cv2.dnn.blobFromImage(imgs,1,(300,300),(104,177,123),swapRB=False,crop=False)
    detector_model.setInput(img_blob)
    detections = detector_model.forward()
    
    if len(detections)>0:
        # i= np.argmax(detections[0,0,:,2])
        # confidence= detections[0,0,i,2]
        for i , confidence in enumerate(detections[0,0,:,2]):
            if confidence>=0.5:
                box= detections[0,0,i,3:7]
                box= box*np.array([w,h,w,h])
                box= box.astype(int)
                sx,sy,ex,ey=box
                cv2.rectangle(imgs,(sx,sy),(ex,ey),(0,255,0),2)
                roi = imgs[sy:ey,sx:ex].copy()
                faceblob= cv2.dnn.blobFromImage(roi,1/255,(96,96),(0,0,0),swapRB=True,crop=True)
                descriptor_model.setInput(faceblob)
                vectors=descriptor_model.forward()

                face_name= face_recognition_model.predict(vectors)[0]
                # face_score =face_recognition_model.predict_proba(vectors).max()

                emotion_name= emotion_recognition_model.predict(vectors)[0]
                emotion_score =emotion_recognition_model.predict_proba(vectors).max()  

                # text_face= '{}:{:.0f}%'.format(face_name,100*face_score)
                text_face= '{}'.format(face_name)
                cv2.putText(imgs,text_face,(sx-50,sy),cv2.FONT_HERSHEY_PLAIN,2,(255,255,255),2)

                text_face= '{}:{:.0f}%'.format(emotion_name,100*emotion_score)
                cv2.putText(imgs,text_face,(sx-50,ey),cv2.FONT_HERSHEY_PLAIN,2,(255,255,255),2)
                # print(face_name)
                # print(face_score)

    return imgs




In [13]:
img_pred=helper('./images/Will_Smith/002_078f6fe5.jpg')
img_pred1=helper('./images/Tom Hanks/002_f6b26479.jpg')
img_pred2=helper('./images/Megan_Fox/004_6aede3d3.jpg')

cv2.imshow('predict',img_pred)
cv2.imshow('predict1',img_pred1)
cv2.imshow('predict2',img_pred2)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [6]:
data= dict(data=[],label=[])